# Agentic Law: data pipeline and knowledge graph
Yashwardhan Ramchaware · Group 12, Project 15

This notebook runs the shared implementation and inspects its outputs. The archive already contains a complete metadata/graph run. The judgment PDFs are not attached: text outputs remain empty until the corpus is added. Read `docs/HANDOFF.md` before training or freezing a benchmark.

For Colab, upload the supplied ZIP, extract it using the Files panel or `zipfile.ZipFile(...).extractall(...)`, and set `PROJECT_ROOT` below to the extracted project folder. No API key is needed.

In [ ]:
from pathlib import Path
import sys, json, subprocess

# Change this if the notebook kernel starts in another directory.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / 'run_pipeline.py').is_file(), 'Set PROJECT_ROOT to the extracted project folder.'
print(PROJECT_ROOT)


## 1. Dependencies
Enable the next switch only when packages are missing. Development dependencies add ReportLab for synthetic PDF tests.

In [ ]:
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r',
                    str(PROJECT_ROOT / 'requirements-dev.txt')], check=True)


## 2. Run data preparation
Leave `PDF_ROOT` as `None` for the supplied metadata. To add judgments, set it to the directory containing year folders. An optional manifest maps authoritative document IDs to PDF paths, including text-only cases not in the two metadata subsets. Do not use the proposal PDF as a judgment.

In [ ]:
PDF_ROOT = None  # Example: Path('/content/drive/MyDrive/supreme_court_judgments')
PDF_MANIFEST = None  # Example: Path('/content/mapping.json')
command = [sys.executable, 'run_pipeline.py', '--raw-dir', 'data/raw',
           '--out-dir', 'data/processed', '--cutoff-year', '2018']
if PDF_ROOT is not None:
    command += ['--pdf-root', str(PDF_ROOT)]
if PDF_MANIFEST is not None:
    command += ['--pdf-manifest', str(PDF_MANIFEST)]
completed = subprocess.run(command, cwd=PROJECT_ROOT, text=True, capture_output=True, check=True)
print(completed.stdout)


## 3. Inspect the data-quality report
Observed metadata: 7,493 source rows, 7,101 distinct citation-source IDs, 6,954 land IDs, and 3,850 overlapping IDs. The proposal listed 7,497 rows. The graph includes external statutes/sections/judgments, so its 128,436 nodes are not all verified Supreme Court cases.

In [ ]:
OUTPUT = PROJECT_ROOT / 'data/processed'
quality = json.loads((OUTPUT / 'quality_report.json').read_text(encoding='utf-8'))
print(json.dumps(quality, indent=2, ensure_ascii=False))


## 4. Consume the shared contracts
Jhon uses sentences/chunks for models and retrieval, Anushka uses chunks and historical graph expansion, and Tarandeep uses split/benchmark artifacts. Keep `doc_id` as a string; keep every case together in one data partition.

In [ ]:
def read_jsonl(path):
    with Path(path).open(encoding='utf-8') as stream:
        yield from (json.loads(line) for line in stream if line.strip())
cases = {r['doc_id']: r for r in read_jsonl(OUTPUT / 'cases.jsonl')}
print('Known cases:', len(cases))
print('Cases with usable text:', sum(r['has_text'] for r in cases.values()))
print('Example:', next(iter(cases.values())))
print('Split status:', json.loads((OUTPUT / 'split_manifest.json').read_text())['status'])


## 5. Explore prior citation neighbors
This is graph inspection. For RAG evidence set `require_text=True`; with this metadata-only snapshot there are no text-bearing neighbors. Use retrieved prior-case IDs as seeds and exclude the held-out query. Unknown years and future nodes fail the cutoff at every hop.

In [ ]:
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from agentic_law_data.graph import Graph
graph = Graph.from_directory(OUTPUT / 'graph')
seeds = [i for i, r in cases.items() if r['year'] is not None and r['year'] < 2018][:5]
neighbors = graph.expand(seeds, direction='outgoing', hops=1, as_of_year=2018,
                         require_text=False, exclude_ids=[], max_nodes=10)
print([(r['doc_id'], r['year'], r['case_name']) for r in neighbors])


## 6. Run tests
These tests check citation orientation, duplicate merging, type uncertainty, temporal leakage, frozen query identity, PDF mapping, real synthetic-PDF extraction, redaction limits, and exact chunk offsets. They do not measure legal-answer accuracy.

In [ ]:
result = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'],
                        cwd=PROJECT_ROOT, text=True, capture_output=True, check=True)
print(result.stdout + result.stderr)


## Team handoff
- `docs/HANDOFF.md`: schemas, integration examples, temporal evaluation, Git workflow.
- `docs/IMPLEMENTATION_REPORT.md`: report-ready account of completed work and current limits.
- `neo4j/README.md`: optional graph database import; live server import was not tested here.

Full-corpus extraction and manual label/query review remain pending the judgment PDFs. Silver citations are incomplete relevance labels and do not establish support for a generated claim.